## Shivam Hippalgave

# Assignment - 6 part - 1

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import time
import random

# --- 1. Dataset Generation (Sequence Reversal) ---
# Task: The model must learn to reverse a sequence of numbers (e.g., 1, 5, 2 -> 2, 5, 1).
# This stresses the "information bottleneck" of the Baseline model.
SEQ_LEN = 15
VOCAB_SIZE = 25
NUM_SAMPLES = 1500

def generate_data():
    X = torch.randint(1, VOCAB_SIZE, (NUM_SAMPLES, SEQ_LEN))
    Y = torch.flip(X, [1]) # Target is the reversed sequence
    return X, Y

X_train, Y_train = generate_data()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Dataset generated. Using device: {device}")

Dataset generated. Using device: cpu


In [6]:
class BiLSTMEncoder(nn.Module):
    def __init__(self, emb_dim, hid_dim):
        super().__init__()
        self.embedding = nn.Embedding(VOCAB_SIZE, emb_dim)
        self.rnn = nn.LSTM(emb_dim, hid_dim, bidirectional=True, batch_first=True)
        self.fc = nn.Linear(hid_dim * 2, hid_dim)

    def forward(self, src):
        embedded = self.embedding(src)
        outputs, (hidden, cell) = self.rnn(embedded)
        # Merge bidirectional states
        hidden = torch.tanh(self.fc(torch.cat((hidden[-2,:,:], hidden[-1,:,:]), dim=1)))
        return outputs, hidden

In [7]:
class Attention(nn.Module):
    def __init__(self, hid_dim):
        super().__init__()
        self.attn = nn.Linear(hid_dim * 3, hid_dim)
        self.v = nn.Linear(hid_dim, 1, bias=False)

    def forward(self, hidden, encoder_outputs):
        seq_len = encoder_outputs.shape[1]
        hidden_repeated = hidden.unsqueeze(1).repeat(1, seq_len, 1)
        energy = torch.tanh(self.attn(torch.cat((hidden_repeated, encoder_outputs), dim=2)))
        attention_scores = self.v(energy).squeeze(2)
        return F.softmax(attention_scores, dim=1)

In [8]:
class Decoder(nn.Module):
    def __init__(self, emb_dim, hid_dim, use_attention=False):
        super().__init__()
        self.use_attention = use_attention
        self.embedding = nn.Embedding(VOCAB_SIZE, emb_dim)
        self.attention = Attention(hid_dim) if use_attention else None

        rnn_input_dim = emb_dim + (hid_dim * 2 if use_attention else 0)
        self.rnn = nn.GRU(rnn_input_dim, hid_dim, batch_first=True)

        # When use_attention=False, this equals hid_dim(32) + emb_dim(16) = 48
        fc_input_dim = hid_dim + (hid_dim * 2 if use_attention else 0) + emb_dim
        self.fc_out = nn.Linear(fc_input_dim, VOCAB_SIZE)

    def forward(self, input, hidden, encoder_outputs):
        embedded = self.embedding(input.unsqueeze(1))

        if self.use_attention:
            a = self.attention(hidden, encoder_outputs).unsqueeze(1)
            context = torch.bmm(a, encoder_outputs)
            rnn_input = torch.cat((embedded, context), dim=2)
            output, hidden = self.rnn(rnn_input, hidden.unsqueeze(0))
            prediction = self.fc_out(torch.cat((output.squeeze(1), context.squeeze(1), embedded.squeeze(1)), dim=1))
        else:
            output, hidden = self.rnn(embedded, hidden.unsqueeze(0))
            # FIX: Removed context_pad here so we concatenate exactly 32 + 16 = 48 features
            prediction = self.fc_out(torch.cat((output.squeeze(1), embedded.squeeze(1)), dim=1))

        return prediction, hidden.squeeze(0)

In [9]:
# --- 3. Training & Evaluation Pipeline ---
def train_and_evaluate(use_attention, epochs=40):
    encoder = BiLSTMEncoder(16, 32).to(device)
    decoder = Decoder(16, 32, use_attention).to(device)

    optimizer = optim.Adam(list(encoder.parameters()) + list(decoder.parameters()), lr=0.01)
    criterion = nn.CrossEntropyLoss()

    dataset_src = X_train.to(device)
    dataset_trg = Y_train.to(device)

    start_time = time.time()

    for epoch in range(epochs):
        optimizer.zero_grad()
        encoder_outputs, hidden = encoder(dataset_src)

        loss = 0
        correct = 0
        total = 0

        for t in range(SEQ_LEN):
            # Teacher forcing
            input_token = dataset_src[:, 0] if t == 0 else dataset_trg[:, t-1]
            output, hidden = decoder(input_token, hidden, encoder_outputs)
            loss += criterion(output, dataset_trg[:, t])

            predictions = output.argmax(dim=1)
            correct += (predictions == dataset_trg[:, t]).sum().item()
            total += dataset_trg.shape[0]

        loss.backward()
        optimizer.step()

    train_time = time.time() - start_time
    accuracy = (correct / total) * 100

    # Auto-generate qualitative analysis based on numerical accuracy
    if accuracy > 90:
        quality = "Excellent sequence alignment; retains full context."
    elif accuracy > 50:
        quality = "Moderate alignment; partial information loss."
    else:
        quality = "Severe information loss; fails to map long sequences."

    return loss.item(), accuracy, train_time, quality

print("Training Baseline Model (Without Attention)... Please wait.")
base_loss, base_acc, base_time, base_qual = train_and_evaluate(use_attention=False)

print("Training Attention Model... Please wait.")
attn_loss, attn_acc, attn_time, attn_qual = train_and_evaluate(use_attention=True)


print("\n" + "="*80)
print("PART 3: AUTOMATED COMPARISON TABLE (COPY/PASTE INTO REPORT)")
print("="*80 + "\n")

print("| Metric | Without Attention | With Attention |")
print("| :--- | :--- | :--- |")
print(f"| **Loss** | {base_loss:.4f} | {attn_loss:.4f} |")
print(f"| **Accuracy / BLEU (Equivalent)** | {base_acc:.2f}% | {attn_acc:.2f}% |")
print(f"| **Training Time** | {base_time:.2f} seconds | {attn_time:.2f} seconds |")
print(f"| **Output Quality** | {base_qual} | {attn_qual} |")

print("\n" + "="*80)

Training Baseline Model (Without Attention)... Please wait.
Training Attention Model... Please wait.

PART 3: AUTOMATED COMPARISON TABLE (COPY/PASTE INTO REPORT)

| Metric | Without Attention | With Attention |
| :--- | :--- | :--- |
| **Loss** | 41.7147 | 27.6914 |
| **Accuracy / BLEU (Equivalent)** | 16.77% | 40.07% |
| **Training Time** | 6.03 seconds | 17.88 seconds |
| **Output Quality** | Severe information loss; fails to map long sequences. | Severe information loss; fails to map long sequences. |

